# JAX-RANS quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovannicatalani/jax-rans/blob/main/notebooks/quickstart.ipynb)

A transonic airfoil in a few minutes: the Euler equations round a NACA 0012 at
Mach 0.75 and 2 degrees, solved to round-off on a generated grid, then the
derivatives of its lift and drag with respect to the angle of attack and the
Mach number, checked against finite differences.

On Colab, choose a GPU runtime first (*Runtime → Change runtime type → T4 GPU*).
The CPU runtime works too, about ten times slower.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("jaxrans") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "jax-rans[plot] @ git+https://github.com/giovannicatalani/jax-rans"])

In [ ]:
import jax
import jaxrans                    # enables float64 for the whole session

print("JAX-RANS", jaxrans.__version__, "on", jax.devices())

## The case

A case is two documents, the physics and the numerics; here they are written
in Python instead of the usual `case.json` and `numerical.json`
([case reference](https://giovannicatalani.github.io/jax-rans/case_reference/)).
The O-grid is generated round the airfoil. The limiter is `van_albada_smooth`,
which has no kink, so that the solution is differentiable; the run goes to round-off
(`forces_tol` null) because derivatives need a converged state.

In [ ]:
from jaxrans.case import parse_case

case = parse_case(
    {"general": {"case_name": "naca0012_transonic"},
     "domain": {"grid": "o_grid", "airfoil": "naca0012", "cells_airfoil": 96, "cells_normal": 48,
                "wall_spacing": 0.001, "far_field": 50.0},
     "freestream": {"mach": 0.75, "alpha_deg": 2.0}},
    {"convective_fluxes": {"riemann_solver": "hllc", "limiter": "van_albada_smooth"},
     "time_integration": {"solver": "implicit"},
     "convergence": {"max_iters": 3000, "tol": 1e-10, "forces_tol": None, "block_iters": 50}})
grid = case.grid()
print(grid.cells, "cells")

## Solve

In [ ]:
from jaxrans import run

def progress(iteration, residual, orders, forces):
    print(f"{iteration:6d}  residual {residual:.2e} ({orders:5.2f} orders)  CL {forces.cl:+.5f}  CD {forces.cd:.5f}")

result = run(case, grid, progress=progress)
print(result.history.summary())
print(result.forces.summary())
print(result.timing.summary())

Mach number (the shock on the upper surface) and wall pressure:

In [ ]:
from IPython.display import Image, display
from jaxrans.plot import plot_result

directory = result.write("results/naca0012_transonic")
for path in plot_result(directory):
    if path.stem in ("mach", "cp"):
        display(Image(str(path), width=640))

## Derivatives

At a converged state `R(U*(p), p) = 0` for every freestream `p`, so the state
changes by `dU/dp = -(dR/dU)^-1 ∂R/∂p`, one linear solve per parameter, with
the exact Jacobian of the residual that was converged; every force
coefficient then follows ([method](https://giovannicatalani.github.io/jax-rans/gradients/)).

In [ ]:
from jaxrans import sensitivity

s = sensitivity(result, outputs=("cl", "cd"), parameters=("alpha", "mach"))
print(s.summary())

## Check against finite differences

Two more converged runs at alpha ± 0.0025 degrees, run together as one batch
(`run_polar`), give the central difference; a step this small keeps its
truncation error, which the shock makes large, below 1e-5.

In [ ]:
from jaxrans.polar import run_polar

h = 0.0025
polar = run_polar(case, [case.flow.alpha_deg - h, case.flow.alpha_deg + h], grid)
minus, plus = polar.results
for name in ("cl", "cd"):
    fd = (getattr(plus.forces, name) - getattr(minus.forces, name)) / (2 * h)
    exact = s.derivatives[name]["alpha"]
    print(f"d{name.upper()}/dalpha: {exact:+.8f} (sensitivity)  {fd:+.8f} (finite difference)  "
          f"relative difference {abs(exact - fd) / abs(fd):.1e}")

## Next

- Your own airfoil from a coordinate file, a multi-element section or a 3D
  wing: the [documentation](https://giovannicatalani.github.io/jax-rans/).
- The same from the shell: `jaxrans CASE_DIR` with the two JSON files
  ([examples](https://github.com/giovannicatalani/jax-rans/tree/main/examples)).